# `18 — Hash table`

**Definition**: `Hash Table` is a data structure that represents a dynamic set of data. It supports `insert`, `delete` and `search` operations. Where it really shine is search, wehre the average search time is $O(1)$, althought this degrades to $O(N)$ in the worst-case


- *dictionary*: Generic way to map keys to values
- *hash table*: Implementation of a dictionary using a hash function

<div style="text-align: center;">
    <img src="./Images/Screenshot 2026-01-13 at 10.32.01.png" width="1000" alt="Screenshot">
</div>

We want an associative array:
- insert(k, v)
- get(k)
- remove(k)

We store items in an array of size $M$ using hash $h(k)$ in $[0..M-1]$.
Because collisions exist, we must resolve them.

Define load factor:
$$\alpha = \frac{N}{M}$$
where $N$ is the number of stored keys.

---

## **Separate chaining**
Each bucket stores a list of (key,value) pairs.
Operations traverse only that bucket.

Expected chain length is:
$$E[L] = O(\alpha + 1)$$
So expected time per operation: $O(\alpha + 1)$.
Keep $\alpha = O(1)$ using rehashing → amortized $O(1)$.

<div style="display: flex; justify-content: center; gap: 10px;">
    <img src="./Images/Screenshot 2026-01-13 at 10.38.02.png" style="width: 49%; height: auto;" alt="Screenshot 1">
    <img src="./Images/Screenshot 2026-01-13 at 10.38.56.png" style="width: 49%; height: auto;" alt="Screenshot 2">
</div>

<br>

<div style="display: flex; justify-content: center; gap: 10px;">
    <img src="./Images/Screenshot 2026-01-13 at 10.42.45.png" style="width: 49%; height: auto;" alt="Screenshot 1">
    <img src="./Images/Screenshot 2026-01-13 at 10.43.52.png" style="width: 49%; height: auto;" alt="Screenshot 2">
</div>

In [ ]:
# Separate chaining hash table with visualization:
from dataclasses import dataclass
from typing import Any, List, Optional, Tuple, Self


@dataclass
class HashTableChaining:
    m: int
    buckets: List[List[Tuple[Any, Any]]]

    @classmethod
    def build(cls, *, m: int) -> "HashTableChaining":
        return cls(m=m, buckets=[[] for _ in range(m)])

    def _h(self: Self, k: Any) -> int:
        return hash(k) % self.m

    def show(self: Self, label: str = "") -> None:
        if label:
            print(label)
        for i, b in enumerate(self.buckets):
            print(f"{i:>2}: {b}")

    def insert(self: Self, *, k: Any, v: Any, verbose: bool = True) -> None:
        idx: int = self._h(k)
        if verbose:
            print("-" * 70)
            print(f"insert(k={k!r}, v={v!r}) -> bucket {idx}")
        bucket = self.buckets[idx]

        for j, (kk, vv) in enumerate(bucket):
            if kk == k:
                bucket[j] = (k, v)
                if verbose:
                    print("  key exists -> update")
                    self.show(label="  buckets:")
                return

        bucket.append((k, v))
        if verbose:
            print("  key new -> append")
            self.show(label="  buckets:")

    def get(self: Self, *, k: Any, verbose: bool = True) -> Optional[Any]:
        idx: int = self._h(k)
        bucket = self.buckets[idx]
        if verbose:
            print("-" * 70)
            print(f"get(k={k!r}) -> bucket {idx} scan length {len(bucket)}")
        for kk, vv in bucket:
            if kk == k:
                if verbose:
                    print("  found:", vv)
                return vv
        if verbose:
            print("  not found")
        return None

    def remove(self: Self, *, k: Any, verbose: bool = True) -> bool:
        idx: int = self._h(k)
        bucket = self.buckets[idx]
        if verbose:
            print("-" * 70)
            print(f"remove(k={k!r}) -> bucket {idx}")
        for j, (kk, vv) in enumerate(bucket):
            if kk == k:
                bucket.pop(j)
                if verbose:
                    print("  removed")
                    self.show(label="  buckets:")
                return True
        if verbose:
            print("  not found")
        return False


ht = HashTableChaining.build(m=5)
ht.insert(k="apple", v=10)
ht.insert(k="banana", v=7)
ht.insert(k="orange", v=3)
ht.insert(k="apple", v=99)   # update
_ = ht.get(k="banana")
_ = ht.remove(k="orange")

----------------------------------------------------------------------
insert(k='apple', v=10) -> bucket 2
  key new -> append
  buckets:
 0: []
 1: []
 2: [('apple', 10)]
 3: []
 4: []
----------------------------------------------------------------------
insert(k='banana', v=7) -> bucket 0
  key new -> append
  buckets:
 0: [('banana', 7)]
 1: []
 2: [('apple', 10)]
 3: []
 4: []
----------------------------------------------------------------------
insert(k='orange', v=3) -> bucket 4
  key new -> append
  buckets:
 0: [('banana', 7)]
 1: []
 2: [('apple', 10)]
 3: []
 4: [('orange', 3)]
----------------------------------------------------------------------
insert(k='apple', v=99) -> bucket 2
  key exists -> update
  buckets:
 0: [('banana', 7)]
 1: []
 2: [('apple', 99)]
 3: []
 4: [('orange', 3)]
----------------------------------------------------------------------
get(k='banana') -> bucket 0 scan length 1
  found: 7
----------------------------------------------------------------

<div style="text-align: center;">
    <video width="1000" controls>
    <source src="./Videos/HashTableChainingVisuals.mp4" type="video/mp4">
    </video>
</div>

## **Open addressing**
Store (key,value) directly in table slots.
If collision occurs, probe other slots (linear probing, double hashing).

Works fast when $\alpha$ is small,
but becomes slow as $\alpha$ approaches $1$.
So we maintain $\alpha$ relatively small.

In [ ]:
# Open addressing (linear probing) with visualization (probe sequence)
from dataclasses import dataclass
from typing import Any, List, Optional, Self


_TOMBSTONE = object()


@dataclass
class HashTableLinearProbing:
    m: int
    keys: List[Optional[Any]]
    values: List[Optional[Any]]
    size: int = 0

    @classmethod
    def build(cls, *, m: int) -> "HashTableLinearProbing":
        return cls(m=m, keys=[None]*m, values=[None]*m)

    def _h(self: Self, k: Any) -> int:
        return hash(k) % self.m

    def show(self: Self, label: str = "") -> None:
        if label:
            print(label)
        for i in range(self.m):
            k = self.keys[i]
            v = self.values[i]
            if k is None:
                print(f"{i:>2}: EMPTY")
            elif k is _TOMBSTONE:
                print(f"{i:>2}: TOMBSTONE")
            else:
                print(f"{i:>2}: {k!r} -> {v!r}")

    def insert(self: Self, *, k: Any, v: Any, verbose: bool = True) -> None:
        if self.size >= self.m:
            raise RuntimeError("table full (need rehash)")

        i: int = self._h(k)
        first_tomb: Optional[int] = None

        if verbose:
            print("-" * 70)
            print(f"insert(k={k!r}, v={v!r}) start i={i}")

        steps: int = 0
        while True:
            if verbose:
                print(f"  probe i={i}")

            if self.keys[i] is None:
                target = first_tomb if first_tomb is not None else i
                if self.keys[target] is None or self.keys[target] is _TOMBSTONE:
                    self.keys[target] = k
                    self.values[target] = v
                    self.size += 1
                if verbose:
                    print(f"  placed at {target}")
                    self.show(label="  table:")
                return

            if self.keys[i] is _TOMBSTONE:
                if first_tomb is None:
                    first_tomb = i

            elif self.keys[i] == k:
                self.values[i] = v
                if verbose:
                    print("  key exists -> update")
                    self.show(label="  table:")
                return

            i = (i + 1) % self.m
            steps += 1
            if steps > self.m:
                raise RuntimeError("no free slot found")

    def get(self: Self, *, k: Any, verbose: bool = True) -> Optional[Any]:
        i: int = self._h(k)
        if verbose:
            print("-" * 70)
            print(f"get(k={k!r}) start i={i}")

        steps: int = 0
        while True:
            if verbose:
                print(f"  probe i={i}")

            if self.keys[i] is None:
                if verbose:
                    print("  hit EMPTY -> not found")
                return None

            if self.keys[i] is not _TOMBSTONE and self.keys[i] == k:
                if verbose:
                    print("  found:", self.values[i])
                return self.values[i]

            i = (i + 1) % self.m
            steps += 1
            if steps > self.m:
                return None

    def remove(self: Self, *, k: Any, verbose: bool = True) -> bool:
        i: int = self._h(k)
        if verbose:
            print("-" * 70)
            print(f"remove(k={k!r}) start i={i}")

        steps: int = 0
        while True:
            if verbose:
                print(f"  probe i={i}")

            if self.keys[i] is None:
                if verbose:
                    print("  hit EMPTY -> not found")
                return False

            if self.keys[i] is not _TOMBSTONE and self.keys[i] == k:
                self.keys[i] = _TOMBSTONE
                self.values[i] = None
                self.size -= 1
                if verbose:
                    print(f"  removed -> placed TOMBSTONE at {i}")
                    self.show(label="  table:")
                return True

            i = (i + 1) % self.m
            steps += 1
            if steps > self.m:
                return False


ht2 = HashTableLinearProbing.build(m=7)
ht2.insert(k="apple", v=10)
ht2.insert(k="banana", v=7)
ht2.insert(k="orange", v=3)
_ = ht2.get(k="banana")
_ = ht2.remove(k="banana")
_ = ht2.get(k="banana")
ht2.insert(k="grape", v=5)  # may reuse tombstone depending on probes

----------------------------------------------------------------------
insert(k='apple', v=10) start i=3
  probe i=3
  placed at 3
  table:
 0: EMPTY
 1: EMPTY
 2: EMPTY
 3: 'apple' -> 10
 4: EMPTY
 5: EMPTY
 6: EMPTY
----------------------------------------------------------------------
insert(k='banana', v=7) start i=0
  probe i=0
  placed at 0
  table:
 0: 'banana' -> 7
 1: EMPTY
 2: EMPTY
 3: 'apple' -> 10
 4: EMPTY
 5: EMPTY
 6: EMPTY
----------------------------------------------------------------------
insert(k='orange', v=3) start i=0
  probe i=0
  probe i=1
  placed at 1
  table:
 0: 'banana' -> 7
 1: 'orange' -> 3
 2: EMPTY
 3: 'apple' -> 10
 4: EMPTY
 5: EMPTY
 6: EMPTY
----------------------------------------------------------------------
get(k='banana') start i=0
  probe i=0
  found: 7
----------------------------------------------------------------------
remove(k='banana') start i=0
  probe i=0
  removed -> placed TOMBSTONE at 0
  table:
 0: TOMBSTONE
 1: 'orange' -> 3
 2

<div style="text-align: center;">
    <video width="1000" controls>
    <source src="./Videos/LinearProbingVisuals.mp4" type="video/mp4">
    </video>
</div>

## **Final summary:**

### **Hashing**
- Expected collisions for random function: **E = N/M**
- Universal family guarantees: **P(collision) ≤ 1/M**
- Polynomial hash supports fast substring hashing.

### **Rabin–Karp**
- Rolling hash compares substring hashes in O(1)
- Total expected: **O(|s| + |t| + verification work)**

### **Hash tables**
| Method | Collision handling | Expected op time (α=N/M) | Notes |
|---|---|---:|---|
| Separate chaining | linked list per bucket | **O(α+1)** | easy deletes |
| Open addressing | probing in array | depends strongly on α | keep α small; Python uses open addressing + double hashing |